# FedMamba-SALT: Centralized Pre-training Experiment

This notebook runs the **full centralized FedMamba-SALT experiment** on Google Colab Pro.

**Pipeline overview:**
1. Environment setup & dependency installation
2. MAE checkpoint verification
3. Component smoke tests
4. Dataset inspection & augmentation visualization
5. Centralized SALT pre-training (100 epochs)
6. Training diagnostics & visualization (loss curve, embedding std, LR, GPU memory)
7. Linear probe evaluation
8. Full fine-tuning evaluation
9. Results summary & Google Drive backup

**Requirements:** Colab Pro with GPU runtime (A100 recommended).

---

## Section 1: Environment Setup

### 1.1 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### 1.2 — Set Paths (⚠️ EDIT THESE)

Update the three paths below to match your Google Drive folder structure.

In [ ]:
# ============================================================
# ⚠️  EDIT THESE THREE PATHS TO MATCH YOUR GOOGLE DRIVE LAYOUT
# ============================================================
DRIVE_REPO    = "/content/drive/MyDrive/FedMamba-SALT"           # repo root on Drive
DRIVE_DATASET = "/content/drive/MyDrive/datasets/retina"         # must have train/ and test/
DRIVE_CKPT    = "/content/drive/MyDrive/FedMamba-SALT/data/ckpts/mae_vit_base.pth"
# ============================================================

NUM_CLASSES = 2  # binary classification

### 1.3 — Copy Repo to Local Colab Filesystem

Copying to `/content/` avoids slow Google Drive I/O on every Python import. The dataset stays on Drive — DataLoader handles large sequential reads efficiently.

In [ ]:
import shutil, os

LOCAL_REPO = "/content/fedmamba_salt"
if os.path.exists(LOCAL_REPO):
    shutil.rmtree(LOCAL_REPO)
shutil.copytree(DRIVE_REPO, LOCAL_REPO)
os.chdir(LOCAL_REPO)
print(f"Repo copied to {LOCAL_REPO}")

# Copy the MAE checkpoint to the expected local path
CKPT_DIR = os.path.join(LOCAL_REPO, "data", "ckpts")
CKPT_PATH = os.path.join(CKPT_DIR, "mae_vit_base.pth")
os.makedirs(CKPT_DIR, exist_ok=True)
if not os.path.exists(CKPT_PATH):
    shutil.copy2(DRIVE_CKPT, CKPT_PATH)
    size_mb = os.path.getsize(CKPT_PATH) / 1e6
    print(f"Checkpoint copied: {size_mb:.1f} MB")
else:
    print(f"Checkpoint already present: {CKPT_PATH}")

# Define output paths used throughout the notebook
OUTPUT_DIR = os.path.join(LOCAL_REPO, "outputs", "retina_centralized")
DRIVE_OUTPUT = os.path.join(DRIVE_REPO, "outputs", "retina_centralized")
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(DRIVE_OUTPUT, exist_ok=True)

### 1.4 — Install Dependencies

In [ ]:
%%capture install_output
# Build tools required for compiling CUDA kernels
!pip install packaging ninja

# Install order matters: causal-conv1d before mamba-ssm
# --no-build-isolation lets the build see the existing PyTorch/CUDA
!pip install causal-conv1d>=1.4.0 --no-build-isolation
!pip install mamba-ssm --no-build-isolation

# Teacher backbone and utilities
!pip install timm==0.3.2
!pip install einops PyYAML matplotlib tqdm pandas scikit-image

In [ ]:
# Show any errors from install (the output is captured above)
install_text = install_output.stdout + install_output.stderr
errors = [line for line in install_text.split('\n') if 'error' in line.lower() or 'failed' in line.lower()]
if errors:
    print("⚠️  Install errors detected:")
    for e in errors:
        print(f"  {e}")
else:
    print("✓ All dependencies installed successfully.")

### 1.5 — Patch timm for PyTorch 2.x Compatibility

`timm==0.3.2` imports `torch._six` which was removed in PyTorch 2.0+. This shim **must** run before any `import timm`.

> **If you restart the runtime**, re-run from this cell onwards.

In [ ]:
import collections.abc, sys, types

# Create a shim module that satisfies timm's internal import
mock_six = types.ModuleType("torch._six")
mock_six.container_abcs = collections.abc
sys.modules["torch._six"] = mock_six

print("✓ timm compatibility patch applied.")

### 1.6 — Verify Environment

In [ ]:
import torch, timm

print(f"PyTorch:    {torch.__version__}")
print(f"CUDA:       {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU:        {torch.cuda.get_device_name(0)}")
    mem_gb = torch.cuda.get_device_properties(0).total_mem / 1e9
    print(f"GPU Memory: {mem_gb:.1f} GB")
else:
    print("⚠️  No GPU detected! Select GPU runtime: Runtime > Change runtime type > A100")
print(f"timm:       {timm.__version__}")

try:
    from mamba_ssm import Mamba
    print(f"mamba-ssm:  ✓ (real CUDA kernels)")
except ImportError as e:
    print(f"mamba-ssm:  ✗ FAILED - {e}")

# Add project to Python path
sys.path.insert(0, LOCAL_REPO)
print(f"\nProject root: {LOCAL_REPO}")

---
## Section 2: Checkpoint Verification

Inspects the raw MAE checkpoint to confirm prefix stripping in `vit_teacher.py` will work correctly.

In [ ]:
!python -m tests.verify_checkpoint --ckpt_path "{CKPT_PATH}"

---
## Section 3: Smoke Tests

Run all component tests before committing GPU hours to training. **All must pass.**

In [ ]:
print("=" * 55)
print("  Running all smoke tests...")
print("=" * 55)
print()

!python -m tests.test_loss
print()
!python -m tests.test_teacher
print()
!python -m tests.test_end_to_end

> ⚠️ **If any test fails, STOP HERE.** Debug the failing component before proceeding to training.

---
## Section 4: Dataset Inspection

### 4.1 — Verify Dataset Structure

In [ ]:
import os, sys
sys.path.insert(0, REPO_DIR)

from augmentations.retina_dataset import RetinaDataset

# Load Retina dataset in SSL-FL format
train_ds = RetinaDataset(
    data_path=DATA_DIR,
    phase='train',
    split_type='central',
    split_csv='train.csv',
)

test_ds = RetinaDataset(
    data_path=DATA_DIR,
    phase='test',
    split_type='central',
    split_csv='test.csv',
)

classes = train_ds.classes
print('=' * 40)
print(f'  Dataset Structure (SSL-FL Retina format)')
print('=' * 40)
print(f'Train: {len(train_ds)} images, {len(classes)} classes')
print(f'Test:  {len(test_ds)} images, {len(test_ds.classes)} classes')
print(f'Classes: {classes}')

assert len(classes) == NUM_CLASSES, f'Expected {NUM_CLASSES} classes, found {len(classes)}'
print(f'\nDataset OK: {NUM_CLASSES} classes (binary)')

### 4.2 — Visualize Augmentation Pairs

The teacher sees a clean view; the student sees a heavily corrupted view. The gap between them is the learning signal.

In [ ]:
from augmentations.medical_aug import DualViewDataset, get_teacher_transform, get_student_transform
from augmentations.medical_aug import RETINA_MEAN, RETINA_STD
import matplotlib.pyplot as plt
import torch

dual_ds = DualViewDataset(
    train_ds,
    teacher_transform=get_teacher_transform(dataset='retina'),
    student_transform=get_student_transform(dataset='retina'),
)

fig, axes = plt.subplots(3, 2, figsize=(8, 12))
fig.suptitle('Teacher (clean) vs Student (augmented)', fontsize=14, fontweight='bold')
mean = torch.tensor(RETINA_MEAN).view(3, 1, 1)
std  = torch.tensor(RETINA_STD).view(3, 1, 1)

for i in range(3):
    idx = i
    t_view, s_view = dual_ds[idx]
    # Denormalize for display
    t_img = (t_view * std + mean).clamp(0, 1).permute(1, 2, 0).numpy()
    s_img = (s_view * std + mean).clamp(0, 1).permute(1, 2, 0).numpy()
    axes[i, 0].imshow(t_img)
    axes[i, 0].set_title(f'Teacher view #{idx}')
    axes[i, 0].axis('off')
    axes[i, 1].imshow(s_img)
    axes[i, 1].set_title(f'Student view #{idx}')
    axes[i, 1].axis('off')

plt.tight_layout()
plt.show()
print(f'DualViewDataset: {len(dual_ds)} paired samples')

---
## Section 5: Centralized Pre-training

This is the main training cell. It runs for 100 epochs and saves:
- `ckpt_latest.pth` after every epoch (for resume)
- `ckpt_epoch_XXXX.pth` every 10 epochs
- `training_metrics.csv` with per-epoch loss, embedding_std, LR, timing, and GPU memory

### 5.1 — Launch Training

In [ ]:
!python train_centralized.py \
    --data_path "{DRIVE_DATASET}" \
    --teacher_ckpt "{CKPT_PATH}" \
    --output_dir "{OUTPUT_DIR}" \
    --epochs 100 \
    --batch_size 128 \
    --lr 1e-3 \
    --weight_decay 0.05 \
    --save_every 10 \
    --num_workers 2 \
    --device cuda

### 5.2 — Backup Checkpoints to Google Drive

**Run this after training completes** (and optionally during training) to protect against Colab disconnects.

In [ ]:
import shutil

os.makedirs(DRIVE_OUTPUT, exist_ok=True)

backed_up = 0
for f in sorted(os.listdir(OUTPUT_DIR)):
    if f.endswith('.pth') or f.endswith('.csv'):
        src = os.path.join(OUTPUT_DIR, f)
        dst = os.path.join(DRIVE_OUTPUT, f)
        shutil.copy2(src, dst)
        size_mb = os.path.getsize(src) / 1e6
        print(f"  ✓ {f} ({size_mb:.1f} MB)")
        backed_up += 1

print(f"\n{backed_up} files backed up to: {DRIVE_OUTPUT}")

---
## Section 6: Training Diagnostics & Visualization

All plots read from `training_metrics.csv`. This section can be re-run from a new Colab session after loading the CSV from Drive.

### 6.0 — Load Metrics

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

# Try local first, fall back to Drive
METRICS_CSV = os.path.join(OUTPUT_DIR, "training_metrics.csv")
if not os.path.exists(METRICS_CSV):
    METRICS_CSV = os.path.join(DRIVE_OUTPUT, "training_metrics.csv")

df = pd.read_csv(METRICS_CSV)
print(f"Loaded {len(df)} epoch records from {METRICS_CSV}")
print()
df.head(10)

### 6.1 — SALT Loss Curve

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

ax.plot(df['epoch'], df['loss'], color='#2196F3', linewidth=2, label='SALT Loss')
ax.axhline(y=0.3, color='#4CAF50', linestyle='--', alpha=0.7, label='Target (< 0.3)')
ax.axhline(y=1.0, color='#FF9800', linestyle=':', alpha=0.5, label='Orthogonal baseline (1.0)')

ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('Loss (1 - cosine similarity)', fontsize=12)
ax.set_title('SALT Loss Curve', fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
ax.set_ylim(bottom=0)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "plot_loss_curve.png"), dpi=150)
plt.show()

print(f"Epoch 1:   loss = {df['loss'].iloc[0]:.4f}")
print(f"Epoch {len(df):>3}:  loss = {df['loss'].iloc[-1]:.4f}")
print(f"Min loss:  {df['loss'].min():.4f} (epoch {df.loc[df['loss'].idxmin(), 'epoch']})")

### 6.2 — Embedding Std (Collapse Detection)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

ax.plot(df['epoch'], df['student_std'], color='#E91E63', linewidth=2, label='Student std')
ax.plot(df['epoch'], df['teacher_std'], color='#9C27B0', linewidth=2, alpha=0.6, label='Teacher std')
ax.axhline(y=0.05, color='#F44336', linestyle='--', linewidth=2, label='Collapse threshold (0.05)')
ax.axhline(y=0.1, color='#FF9800', linestyle=':', alpha=0.5, label='Warning zone (0.1)')

ax.fill_between(df['epoch'], 0, 0.05, color='#F44336', alpha=0.1)

ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('Embedding Standard Deviation', fontsize=12)
ax.set_title('Representation Collapse Monitor', fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
ax.set_ylim(bottom=0)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "plot_embedding_std.png"), dpi=150)
plt.show()

min_std = df['student_std'].min()
min_epoch = df.loc[df['student_std'].idxmin(), 'epoch']
status = "✓ PASS" if min_std > 0.05 else "✗ FAIL - COLLAPSE DETECTED"
print(f"Min student std: {min_std:.4f} at epoch {min_epoch}  [{status}]")

### 6.3 — Learning Rate Schedule

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))

ax.plot(df['epoch'], df['lr'], color='#009688', linewidth=2)

ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('Learning Rate', fontsize=12)
ax.set_title('Learning Rate Schedule (Linear Warmup + Cosine Decay)', fontsize=14, fontweight='bold')
ax.yaxis.set_major_formatter(mticker.ScalarFormatter(useMathText=True))
ax.ticklabel_format(style='sci', axis='y', scilimits=(0, 0))
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "plot_lr_schedule.png"), dpi=150)
plt.show()

### 6.4 — Per-Epoch Training Time

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))

ax.bar(df['epoch'], df['epoch_time_s'], color='#607D8B', alpha=0.7, width=1.0)
mean_time = df['epoch_time_s'].mean()
ax.axhline(y=mean_time, color='#FF5722', linestyle='--',
           label=f"Mean: {mean_time:.1f}s")

ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('Time (seconds)', fontsize=12)
ax.set_title('Per-Epoch Training Time', fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "plot_epoch_timing.png"), dpi=150)
plt.show()

total_hours = df['epoch_time_s'].sum() / 3600
print(f"Total training time: {total_hours:.2f} hours")
print(f"Average per epoch:   {mean_time:.1f}s")

### 6.5 — GPU Memory Usage

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))

ax.plot(df['epoch'], df['gpu_mem_allocated_mb'] / 1024, color='#3F51B5',
        linewidth=2, label='Allocated')
ax.plot(df['epoch'], df['gpu_mem_reserved_mb'] / 1024, color='#FF9800',
        linewidth=2, alpha=0.7, label='Reserved')
ax.plot(df['epoch'], df['gpu_mem_peak_mb'] / 1024, color='#F44336',
        linewidth=1.5, linestyle='--', alpha=0.7, label='Peak')

# Show total GPU memory as reference line
if torch.cuda.is_available():
    total_gb = torch.cuda.get_device_properties(0).total_mem / (1024**3)
    ax.axhline(y=total_gb, color='gray', linestyle=':', alpha=0.5,
               label=f'Total VRAM ({total_gb:.0f} GB)')

ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('GPU Memory (GB)', fontsize=12)
ax.set_title('GPU Memory Usage Over Training', fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
ax.set_ylim(bottom=0)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "plot_gpu_memory.png"), dpi=150)
plt.show()

peak_mb = df['gpu_mem_peak_mb'].max()
print(f"Peak GPU memory: {peak_mb:.0f} MB ({peak_mb / 1024:.1f} GB)")

### 6.6 — Combined Training Dashboard

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('FedMamba-SALT Training Dashboard', fontsize=16, fontweight='bold')

# --- Loss ---
ax = axes[0, 0]
ax.plot(df['epoch'], df['loss'], color='#2196F3', linewidth=2)
ax.axhline(y=0.3, color='#4CAF50', linestyle='--', alpha=0.7, label='Target')
ax.set_ylabel('Loss'); ax.set_title('SALT Loss')
ax.legend(fontsize=8); ax.grid(True, alpha=0.3); ax.set_ylim(bottom=0)

# --- Embedding Std ---
ax = axes[0, 1]
ax.plot(df['epoch'], df['student_std'], color='#E91E63', linewidth=2, label='Student')
ax.plot(df['epoch'], df['teacher_std'], color='#9C27B0', linewidth=1.5, alpha=0.6, label='Teacher')
ax.axhline(y=0.05, color='#F44336', linestyle='--', label='Collapse')
ax.set_ylabel('Std'); ax.set_title('Embedding Std')
ax.legend(fontsize=8); ax.grid(True, alpha=0.3); ax.set_ylim(bottom=0)

# --- Learning Rate ---
ax = axes[1, 0]
ax.plot(df['epoch'], df['lr'], color='#009688', linewidth=2)
ax.set_ylabel('LR'); ax.set_title('Learning Rate'); ax.set_xlabel('Epoch')
ax.grid(True, alpha=0.3)

# --- GPU Memory ---
ax = axes[1, 1]
ax.plot(df['epoch'], df['gpu_mem_allocated_mb'] / 1024, color='#3F51B5',
        linewidth=2, label='Allocated')
ax.plot(df['epoch'], df['gpu_mem_peak_mb'] / 1024, color='#F44336',
        linewidth=1.5, linestyle='--', label='Peak')
ax.set_ylabel('GB'); ax.set_title('GPU Memory'); ax.set_xlabel('Epoch')
ax.legend(fontsize=8); ax.grid(True, alpha=0.3); ax.set_ylim(bottom=0)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "plot_training_dashboard.png"), dpi=150, bbox_inches='tight')
plt.show()

### 6.7 — Final Health Check

In [ ]:
import torch
from objectives.salt_loss import embedding_std
from utils.ckpt_compat import safe_torch_load
from models.inception_mamba import InceptionMambaEncoder

ckpt = torch.load(os.path.join(OUTPUT_DIR, "ckpt_epoch_0010.pth"), map_location="cpu", weights_only=False)
final_loss = ckpt['loss']
final_epoch = ckpt['epoch'] + 1

# Load student and compute embedding std on random data
student = InceptionMambaEncoder(patch_size=16, embed_dim=256, depth=4, out_dim=768)
student.load_state_dict(ckpt['student_state_dict'])
student.eval()

with torch.no_grad():
    dummy = torch.randn(16, 3, 224, 224)
    emb = student(dummy)
    std_val = embedding_std(emb)

peak_mem = df['gpu_mem_peak_mb'].max()
total_time = df['epoch_time_s'].sum() / 3600

print("=" * 55)
print("  Training Health Check")
print("=" * 55)
print(f"  Final epoch:     {final_epoch}")
print(f"  Final loss:      {final_loss:.4f}  {'✓ PASS' if final_loss < 0.3 else '✗ FAIL'} (target < 0.3)")
print(f"  Embedding std:   {std_val:.4f}  {'✓ PASS' if std_val > 0.05 else '✗ FAIL'} (target > 0.05)")
print(f"  Peak GPU memory: {peak_mem:.0f} MB ({peak_mem / 1024:.1f} GB)")
print(f"  Total time:      {total_time:.2f} hours")
print("=" * 55)

---
## Section 7: Linear Probe Evaluation

Freeze the pre-trained encoder, attach a single linear layer, train only that layer on labeled data.

### 7.1 — Run Linear Probe

In [ ]:
EVAL_LP_DIR = os.path.join(OUTPUT_DIR, "eval_linear_probe")
FINAL_CKPT  = os.path.join(OUTPUT_DIR, "ckpt_epoch_0010.pth")

!python -m eval.linear_probe \
    --encoder_ckpt "{FINAL_CKPT}" \
    --data_path "{DRIVE_DATASET}" \
    --num_classes {NUM_CLASSES} \
    --output_dir "{EVAL_LP_DIR}" \
    --epochs 100 \
    --batch_size 256 \
    --lr 5e-2 \
    --mode linear_probe

### 7.2 — Display Confusion Matrix

In [ ]:
from IPython.display import Image as IPImage, display
import glob

# Try to find any confusion matrix image
cm_files = glob.glob(os.path.join(EVAL_LP_DIR, "*confusion*")) + \
           glob.glob(os.path.join(EVAL_LP_DIR, "*.png"))

if cm_files:
    for f in cm_files:
        print(f"Displaying: {os.path.basename(f)}")
        display(IPImage(filename=f, width=500))
else:
    print("No confusion matrix found. Check eval output:")
    if os.path.exists(EVAL_LP_DIR):
        for f in os.listdir(EVAL_LP_DIR):
            print(f"  {f}")
    else:
        print(f"  Directory not found: {EVAL_LP_DIR}")

---
## Section 8: Full Fine-tuning Evaluation

Unfreeze the encoder and train end-to-end. This is the fair comparison against the Fed-MAE baseline (77.43%).

### 8.1 — Run Full Fine-tuning

In [ ]:
EVAL_FT_DIR = os.path.join(OUTPUT_DIR, "eval_full_finetune")

!python -m eval.linear_probe \
    --encoder_ckpt "{FINAL_CKPT}" \
    --data_path "{DRIVE_DATASET}" \
    --num_classes {NUM_CLASSES} \
    --output_dir "{EVAL_FT_DIR}" \
    --epochs 50 \
    --batch_size 256 \
    --lr 1e-3 \
    --mode full_finetune
# Display the generated metrics and visualizations directly in the notebook
from IPython.display import Image, display
import os

output_dir = '/content/fedmamba_salt/outputs/retina_centralized/eval_full_finetune'
curves = os.path.join(output_dir, 'training_curves_full_finetune.png')
cm = os.path.join(output_dir, 'confusion_matrix_full_finetune.png')

print('\n' + '='*60)
print('  TRAINING CURVES')
print('='*60 + '\n')
if os.path.exists(curves):
    display(Image(filename=curves))

print('\n' + '='*60)
print('  CONFUSION MATRIX')
print('='*60 + '\n')
if os.path.exists(cm):
    display(Image(filename=cm))


### 8.2 — Display Confusion Matrix

In [ ]:
cm_files = glob.glob(os.path.join(EVAL_FT_DIR, "*confusion*")) + \
           glob.glob(os.path.join(EVAL_FT_DIR, "*.png"))

if cm_files:
    for f in cm_files:
        print(f"Displaying: {os.path.basename(f)}")
        display(IPImage(filename=f, width=500))
else:
    print("No confusion matrix found. Check eval output:")
    if os.path.exists(EVAL_FT_DIR):
        for f in os.listdir(EVAL_FT_DIR):
            print(f"  {f}")
    else:
        print(f"  Directory not found: {EVAL_FT_DIR}")

---
## Section 9: Results Summary & Drive Backup

### 9.1 — Final Results

In [ ]:
print("=" * 60)
print("  FedMamba-SALT: Centralized Experiment Results")
print("=" * 60)
print()
print("  Pre-training Health:")
print(f"    Final SALT loss:  {final_loss:.4f}  {'✓ PASS' if final_loss < 0.3 else '✗ FAIL'} (target < 0.3)")
print(f"    Embedding std:    {std_val:.4f}  {'✓ PASS' if std_val > 0.05 else '✗ FAIL'} (target > 0.05)")
print(f"    Peak GPU memory:  {peak_mem:.0f} MB ({peak_mem / 1024:.1f} GB)")
print(f"    Total train time: {total_time:.2f} hours")
print()
print(f"  Downstream Evaluation ({NUM_CLASSES}-class, binary):")
print("    Linear Probe accuracy:    see Section 7 output above")
print("    Full Fine-tune accuracy:  see Section 8 output above")
print()
print("  Baseline Comparison:")
print("    Fed-MAE (Retina Split-3, full fine-tune): 77.43%")
print("    (Note: baseline uses 5-class; your experiment is binary)")
print("=" * 60)

### 9.2 — Save All Results to Google Drive

In [ ]:
import shutil

os.makedirs(DRIVE_OUTPUT, exist_ok=True)

# Save eval directories
for subdir in ["eval_linear_probe", "eval_full_finetune"]:
    src = os.path.join(OUTPUT_DIR, subdir)
    dst = os.path.join(DRIVE_OUTPUT, subdir)
    if os.path.exists(src):
        if os.path.exists(dst):
            shutil.rmtree(dst)
        shutil.copytree(src, dst)
        print(f"  ✓ Saved {subdir}/")

# Save plots, metrics, and checkpoints
for f in sorted(os.listdir(OUTPUT_DIR)):
    fp = os.path.join(OUTPUT_DIR, f)
    if os.path.isfile(fp) and (f.endswith('.png') or f.endswith('.csv') or f.endswith('.pth')):
        shutil.copy2(fp, os.path.join(DRIVE_OUTPUT, f))
        print(f"  ✓ Saved {f}")

print(f"\nAll results saved to: {DRIVE_OUTPUT}")
print("These persist even after Colab disconnects.")

---

### Expected `training_metrics.csv` columns

| Column | Description |
|---|---|
| `epoch` | Epoch number (1-indexed) |
| `loss` | Average SALT loss (1 - cosine similarity, range [0, 2]) |
| `student_std` | Average embedding standard deviation (collapse < 0.05) |
| `teacher_std` | Teacher embedding std (reference) |
| `lr` | Current learning rate |
| `epoch_time_s` | Wall-clock time for the epoch (seconds) |
| `gpu_mem_allocated_mb` | CUDA memory actively used (MB) |
| `gpu_mem_reserved_mb` | CUDA memory reserved by allocator (MB) |
| `gpu_mem_peak_mb` | Peak CUDA memory since training start (MB) |